In [ ]:
# Demo

# Overview
- FER2013 dataset (face images + facial expression)
  + Goal: classifying facial exprression
  + Input: a 48x48 image
  + Output: facial expression (angry, disgust, fear, happy, neutral, sad, and surprise)
- Defining an CNN+MLP
- Training and evaluating an CNN+MLP

# Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import time
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('> device:', device)

In [ ]:
# ===============================
#     PyTorch training helpers
# ===============================
class NumpyImageDataset(Dataset):
    """Wraps numpy arrays: images x of shape (N, H, W, C) and one-hot labels y of shape (N, num_classes)."""
    def __init__(self, x, y, transform=None):
        # channels-last (N, H, W, C) ---> channels-first (N, C, H, W)
        self.x = torch.tensor(np.ascontiguousarray(x.transpose(0, 3, 1, 2)), dtype=torch.float32)
        # one-hot vectors ---> class indices
        self.y = torch.tensor(np.argmax(y, axis=1), dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, self.y[idx]


def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over loader: trains when an optimizer is given, otherwise only evaluates. Returns (loss, acc)."""
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, total_correct / total


def fit(model, train_loader, val_loader, criterion, optimizer, epochs, scheduler=None, callbacks=None):
    """Keras-like model.fit. Returns a history dict with keys 'loss', 'acc', 'val_loss', 'val_acc'.
    - scheduler: e.g. torch.optim.lr_scheduler.ReduceLROnPlateau, stepped with val_loss after every epoch
    - callbacks: callables cb(epoch, logs, model) called after every epoch; return True to stop training"""
    history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    callbacks = callbacks or []
    for epoch in range(epochs):
        t0 = time.time()
        loss, acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        logs = {'loss': loss, 'acc': acc, 'val_loss': val_loss, 'val_acc': val_acc}
        for k, v in logs.items():
            history[k].append(v)
        print('Epoch %d/%d - %.1fs - loss: %.4f - acc: %.4f - val_loss: %.4f - val_acc: %.4f - lr: %g'
              % (epoch + 1, epochs, time.time() - t0, loss, acc, val_loss, val_acc, optimizer.param_groups[0]['lr']))
        if scheduler is not None:
            scheduler.step(val_loss)
        stop = False
        for cb in callbacks:
            stop = bool(cb(epoch, logs, model)) or stop
        if stop:
            break
    return history


def evaluate(model, loader, criterion):
    """Like Keras model.evaluate: returns [loss, accuracy]."""
    return list(run_epoch(model, loader, criterion))


def predict(model, x):
    """Like Keras model.predict: x is a numpy array (N, H, W, C); returns softmax probabilities (N, num_classes)."""
    model.eval()
    with torch.no_grad():
        x = torch.tensor(np.ascontiguousarray(x.transpose(0, 3, 1, 2)), dtype=torch.float32, device=device)
        return F.softmax(model(x), dim=1).cpu().numpy()

In [ ]:
num_classes = 7   # angry, disgust, fear, happy, neutral, sad, and surprise

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRbXpRcnl2SjJkdlk?resourcekey=0-7qr7Ro7hgkmFknsN1yParQ&usp=sharing
This source code and npy files MUST be in the same location

In [ ]:
trainingset = np.loadtxt('fer2013_training_onehot.csv', delimiter=',')
testingset = np.loadtxt('fer2013_publictest_onehot.csv', delimiter=',')

trainingset

In [ ]:
n_inputs = 2304
n_classes = 7
img_dim = 48

x_training = trainingset[:, 0:n_inputs]
y_training = trainingset[:, n_inputs:n_inputs + n_classes]

x_testing = testingset[:, 0:n_inputs]
y_testing = testingset[:, n_inputs:n_inputs + n_classes]

x_training = x_training.reshape(x_training.shape[0], 48, 48)
x_training = np.expand_dims(x_training, axis=3)

x_testing = x_testing.reshape(x_testing.shape[0], 48, 48)
x_testing = np.expand_dims(x_testing, axis=3)

# PyTorch datasets: images (N, 48, 48, 1) ---> (N, 1, 48, 48), one-hot labels ---> class indices
train_set = NumpyImageDataset(x_training, y_training)
test_set = NumpyImageDataset(x_testing, y_testing)

# (2) Declare model

In [ ]:
def keras_init(m):
    # Keras default initialization: glorot_uniform weights, zero biases
    # (PyTorch's default init trains noticeably slower with the same SGD/Adam settings)
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class EmoCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # 48x48 face image, input shape: (N, 1, 48, 48)

        # conv, pooling layers + dropout (padding='same' ---> padding=1, padding='valid' ---> padding=0)
        self.conv1_1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)      # -> 48x48
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=0)     # -> 46x46
        self.pool1 = nn.MaxPool2d(kernel_size=2)                       # -> 23x23
        self.conv_dropout1 = nn.Dropout(p=0.25)

        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)    # -> 23x23
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=0)   # -> 21x21
        self.conv2_3 = nn.Conv2d(128, 218, kernel_size=3, padding=1)   # -> 21x21
        self.conv2_4 = nn.Conv2d(218, 218, kernel_size=3, padding=0)   # -> 19x19
        self.pool2 = nn.MaxPool2d(kernel_size=2)                       # -> 9x9
        self.conv_dropout2 = nn.Dropout(p=0.25)

        # similar to the MLP example!
        # matrix ---> vector: 218 x 9 x 9 = 17658 features
        self.flatten = nn.Flatten()

        # FC layers + dropout
        self.fc1 = nn.Linear(218 * 9 * 9, 1024)
        self.fc_dropout1 = nn.Dropout(p=0.5)

        # outputs raw logits: softmax is included in nn.CrossEntropyLoss (and applied in predict)
        self.fc3_7ways_softmax = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = F.relu(self.conv2_4(x))
        x = self.conv_dropout2(self.pool2(x))

        x = self.flatten(x)

        x = self.fc_dropout1(F.relu(self.fc1(x)))
        return self.fc3_7ways_softmax(x)


# define model (Keras model name: 'emo_cnn')
model = EmoCNN(num_classes).to(device)
model.apply(keras_init)

# print model summary
print(model)
print('Total params: {:,}'.format(sum(p.numel() for p in model.parameters())))
print('Trainable params: {:,}'.format(sum(p.numel() for p in model.parameters() if p.requires_grad)))

In [ ]:
# model architecture (replaces the Keras model_to_dot SVG graph)
print(model)

# (3) Train defined model
- Note that the training history including loss and accuracy will be saved in 'history' variable (a dict returned by our fit helper)
- In case your system runs out of memory (OOM), try to decrease batch size

In [ ]:
# declare learning rate, loss function, and model metric
criterion = nn.CrossEntropyLoss()   # Keras 'categorical_crossentropy' (softmax is included)
lr = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# train the model
batch_size = 128
epochs = 15

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, criterion, optimizer, epochs)
print('> training time is %.4f minutes' % ((time.time() - starting_time)/60))

### Training time
- (Times below were measured with the original Keras/TensorFlow version of this notebook)
- Training time with GTX 1080 is about 6.1120 minutes for 15 epochs
- Training time with Core i7 is around 302.6 minutes (~5.04 hours) for 15 epochs

# (4) Evaluate trained model

In [ ]:
score = evaluate(model, test_loader, criterion)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
def get_emotion(ohv):
    indx = np.argmax(ohv)
        
    if indx == 0:
        return 'angry'
    elif indx == 1:
        return 'disgust'
    elif indx == 2:
        return 'fear'
    elif indx == 3:
        return 'happy'
    elif indx == 4:
        return 'sad'
    elif indx == 5:
        return 'surprise'
    elif indx == 6:
        return 'neutral'

In [ ]:
# 2163
img_indx = np.uint32(np.random.rand()*(testingset.shape[0] - 1))
sample = x_testing[img_indx, :]
sample = sample.reshape(48, 48)

pred_cls = predict(model, sample.reshape(1, 48, 48, 1))

plt.imshow(sample, cmap='gray')
plt.show()
print('> testing image index: %d\n> true emotion: %s\n> predicted emotion: %s' % (img_indx, get_emotion(y_testing[img_indx, :]), get_emotion(pred_cls)))